In [8]:
from pyspark.sql import SparkSession
KAFKA_PKG    = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2"
SEDONA_PKG   = "org.apache.sedona:sedona-spark-shaded-4.0_2.13:1.8.1"
GEOTOOLS_PKG = "org.datasyslab:geotools-wrapper:1.8.1-33.1"
ICEBERG_PKG  = "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1"  
AWS_SDK_PKG  = "org.apache.hadoop:hadoop-aws:3.4.1"

spark = SparkSession.builder \
    .appName("NYC_Environmental_Impact") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages",
            f"{KAFKA_PKG},{SEDONA_PKG},{GEOTOOLS_PKG},{ICEBERG_PKG},{AWS_SDK_PKG}") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.apache.sedona.sql.SedonaSqlExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "s3a://data-lake/") \
    .config("spark.hadoop.fs.s3a.endpoint",      "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",    "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",    "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [9]:
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.default.parallelism", "16")

In [12]:
from pyspark.sql.functions import (
    col,
    expr,
    from_json,
    lit,
    to_timestamp,
    udf,
)
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
import h3

KAFKA_BOOTSTRAP = "redpanda:29092"
CHECKPOINT_PATH = "s3a://data-lake/checkpoints/local.db.enriched_traffic_v6"

traffic_schema = StructType([
    StructField("id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("speed", DoubleType(), True),
    StructField("travel_time", DoubleType(), True),
    StructField("link_name", StringType(), True),
    StructField("borough", StringType(), True),
    StructField("from_street", StringType(), True),
    StructField("to_street", StringType(), True),
    StructField("data_as_of", StringType(), True),
    StructField("link_points", StringType(), True),
    StructField("encoded_poly_line", StringType(), True),
])

openaq_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("locality", StringType(), True),
    StructField("timezone", StringType(), True),
    StructField("coordinates", StructType([
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
    ]), True),
    StructField("datetimeFirst", StructType([
        StructField("utc", StringType(), True),
        StructField("local", StringType(), True),
    ]), True),
    StructField("datetimeLast", StructType([
        StructField("utc", StringType(), True),
        StructField("local", StringType(), True),
    ]), True),
    StructField("country", StructType([
        StructField("code", StringType(), True),
        StructField("name", StringType(), True),
    ]), True),
    StructField("owner", StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ]), True),
    StructField("provider", StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ]), True),
    StructField("sensors", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("parameter", StringType(), True),
        StructField("units", StringType(), True),
    ])), True),
])

purpleair_schema = StructType([
    StructField("name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("pm2.5_10minute", DoubleType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
])

weather_schema = StructType([
    StructField("number", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("startTime", StringType(), True),
    StructField("endTime", StringType(), True),
    StructField("isDaytime", BooleanType(), True),
    StructField("temperature", IntegerType(), True),
    StructField("temperatureUnit", StringType(), True),
    StructField("temperatureTrend", StringType(), True),
    StructField("windSpeed", StringType(), True),
    StructField("windDirection", StringType(), True),
    StructField("shortForecast", StringType(), True),
    StructField("detailedForecast", StringType(), True),
    StructField("probabilityOfPrecipitation", StructType([
        StructField("unitCode", StringType(), True),
        StructField("value", IntegerType(), True),
    ]), True),
])

def read_kafka_json(topic_name: str, json_schema: StructType):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic_name)
        .option("startingOffsets", "latest")
        .load()
        .select(
            from_json(col("value").cast("string"), json_schema).alias("payload"),
            col("timestamp").alias("kafka_timestamp")
        )
        .select("payload.*", "kafka_timestamp")
    )

@udf(returnType=StringType())
def latlon_to_h3(lat, lon):
    if lat is None or lon is None:
        return None
    try:
        return h3.latlng_to_cell(float(lat), float(lon), 9)
    except AttributeError:
        return h3.geo_to_h3(float(lat), float(lon), 9)
    except Exception:
        return None

traffic_raw_df = read_kafka_json("nyc_traffic_raw", traffic_schema)
openaq_raw_df = read_kafka_json("nyc_openaq_raw", openaq_schema)
purpleair_raw_df = read_kafka_json("nyc_purpleair_raw", purpleair_schema)
weather_raw_df = read_kafka_json("nyc_weather_raw", weather_schema)

traffic_df = (
    traffic_raw_df
    .withColumn("first_link_point", expr("trim(element_at(split(link_points, ' '), 1))"))
    .withColumn("traffic_lat", expr("cast(element_at(split(first_link_point, ','), 1) as double)"))
    .withColumn("traffic_lon", expr("cast(element_at(split(first_link_point, ','), 2) as double)"))
    .withColumn("traffic_event_ts", to_timestamp(col("data_as_of"), "yyyy-MM-dd'T'HH:mm:ss.SSS").cast(TimestampType()))
    .withColumn("h3_index", latlon_to_h3(col("traffic_lat"), col("traffic_lon")))
    .withWatermark("traffic_event_ts", "15 minutes")
)

openaq_df = (
    openaq_raw_df
    .withColumn("aq_event_ts", to_timestamp(col("datetimeLast.utc")).cast(TimestampType()))
    .withColumn("aq_lat", col("coordinates.latitude"))
    .withColumn("aq_lon", col("coordinates.longitude"))
    .withColumn("h3_index", latlon_to_h3(col("aq_lat"), col("aq_lon")))
    .select(
        lit("openaq").alias("aq_source"),
        col("id").cast(StringType()).alias("aq_location_id"),
        col("name").alias("aq_location_name"),
        col("aq_event_ts"),
        col("aq_lat"),
        col("aq_lon"),
        col("h3_index"),
        lit(None).cast(DoubleType()).alias("aq_pm25_ugm3"),
        lit(None).cast(DoubleType()).alias("aq_temperature"),
        lit(None).cast(DoubleType()).alias("aq_humidity"),
    )
)

purpleair_df = (
    purpleair_raw_df
    .withColumn("aq_event_ts", col("kafka_timestamp").cast(TimestampType()))
    .withColumn("aq_lat", col("latitude"))
    .withColumn("aq_lon", col("longitude"))
    .withColumn("h3_index", latlon_to_h3(col("aq_lat"), col("aq_lon")))
    .select(
        lit("purpleair").alias("aq_source"),
        col("name").alias("aq_location_id"),
        col("name").alias("aq_location_name"),
        col("aq_event_ts"),
        col("aq_lat"),
        col("aq_lon"),
        col("h3_index"),
        col("`pm2.5_10minute`").cast(DoubleType()).alias("aq_pm25_ugm3"),
        col("temperature").cast(DoubleType()).alias("aq_temperature"),
        col("humidity").cast(DoubleType()).alias("aq_humidity"),
    )
)

air_quality_df = (
    openaq_df
    .unionByName(purpleair_df, allowMissingColumns=True)
    .withWatermark("aq_event_ts", "15 minutes")
)

weather_df = (
    weather_raw_df
    .withColumn("weather_event_ts", to_timestamp(col("startTime")).cast(TimestampType()))
    .withColumn("weather_join_key", lit("nyc"))
    .withWatermark("weather_event_ts", "15 minutes")
)

traffic_air_df = (
    traffic_df.alias("t")
    .join(
        air_quality_df.alias("aq"),
        (
            (col("t.h3_index") == col("aq.h3_index")) &
            (col("aq.aq_event_ts") >= col("t.traffic_event_ts") - expr("INTERVAL 1 MINUTE")) &
            (col("aq.aq_event_ts") <= col("t.traffic_event_ts") + expr("INTERVAL 1 MINUTE"))
        ),
        "leftOuter"
    )
    .select(
        col("t.id").alias("id"),
        col("t.status").alias("status"),
        col("t.speed").alias("speed"),
        col("t.travel_time").alias("travel_time"),
        col("t.link_name").alias("link_name"),
        col("t.borough").alias("borough"),
        col("t.from_street").alias("from_street"),
        col("t.to_street").alias("to_street"),
        col("t.traffic_event_ts"),
        col("t.traffic_lat"),
        col("t.traffic_lon"),
        col("t.h3_index"),
        col("aq.aq_source"),
        col("aq.aq_location_id"),
        col("aq.aq_location_name"),
        col("aq.aq_pm25_ugm3"),
        col("aq.aq_temperature"),
        col("aq.aq_humidity"),
        col("aq.aq_event_ts").cast("string").alias("aq_event_ts_str"),
    )
    .withColumn("weather_join_key", lit("nyc"))
)

enriched_traffic_df = (
    traffic_air_df.alias("ta")
    .join(
        weather_df.alias("w"),
        (
            (col("ta.weather_join_key") == col("w.weather_join_key")) &
            (col("w.weather_event_ts") >= col("ta.traffic_event_ts") - expr("INTERVAL 1 MINUTE")) &
            (col("w.weather_event_ts") <= col("ta.traffic_event_ts") + expr("INTERVAL 1 MINUTE"))
        ),
        "leftOuter"
    )
    .select(
        col("ta.id").alias("traffic_id"),
        col("ta.status").alias("traffic_status"),
        col("ta.speed").alias("traffic_speed"),
        col("ta.travel_time").alias("traffic_travel_time"),
        col("ta.link_name").alias("traffic_link_name"),
        col("ta.borough").alias("traffic_borough"),
        col("ta.from_street").alias("traffic_from_street"),
        col("ta.to_street").alias("traffic_to_street"),
        col("ta.traffic_event_ts"),
        col("ta.traffic_lat"),
        col("ta.traffic_lon"),
        col("ta.h3_index"),
        col("ta.aq_source"),
        col("ta.aq_location_id"),
        col("ta.aq_location_name"),
        col("ta.aq_pm25_ugm3"),
        col("ta.aq_temperature"),
        col("ta.aq_humidity"),
        to_timestamp(col("ta.aq_event_ts_str")).alias("aq_event_ts"),
        col("w.name").alias("weather_period_name"),
        col("w.temperature").alias("weather_temperature"),
        col("w.temperatureUnit").alias("weather_temperature_unit"),
        col("w.windSpeed").alias("weather_wind_speed"),
        col("w.windDirection").alias("weather_wind_direction"),
        col("w.shortForecast").alias("weather_short_forecast"),
        col("w.probabilityOfPrecipitation.value").alias("weather_precip_probability"),
        col("w.weather_event_ts"),
    )
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.db.enriched_traffic (
    traffic_id STRING,
    traffic_status STRING,
    traffic_speed DOUBLE,
    traffic_travel_time DOUBLE,
    traffic_link_name STRING,
    traffic_borough STRING,
    traffic_from_street STRING,
    traffic_to_street STRING,
    traffic_event_ts TIMESTAMP,
    traffic_lat DOUBLE,
    traffic_lon DOUBLE,
    h3_index STRING,
    aq_source STRING,
    aq_location_id STRING,
    aq_location_name STRING,
    aq_pm25_ugm3 DOUBLE,
    aq_temperature DOUBLE,
    aq_humidity DOUBLE,
    aq_event_ts TIMESTAMP,
    weather_period_name STRING,
    weather_temperature INT,
    weather_temperature_unit STRING,
    weather_wind_speed STRING,
    weather_wind_direction STRING,
    weather_short_forecast STRING,
    weather_precip_probability INT,
    weather_event_ts TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(traffic_event_ts), h3_index)
""")

# enriched_traffic_query = (
#     enriched_traffic_df.writeStream
#     .format("iceberg")
#     .outputMode("append")
#     .option("checkpointLocation", CHECKPOINT_PATH)
#     .toTable("local.db.enriched_traffic")
# )
debug_query = (
    enriched_traffic_df.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 20)
    .option("checkpointLocation", "s3a://data-lake/checkpoints/debug_console_v6")
    .start()
)



In [11]:
for q in spark.streams.active:
    print("Stopping:", q.name, q.id)
    q.stop()

spark.streams.active


[]

In [ ]:
import time

# Query the Iceberg table using standard SQL
result = spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 5
""")

result.show()

+----------------+---------------+-------------+------------+-------------------+
|traffic_event_ts|traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+----------------+---------------+-------------+------------+-------------------+
+----------------+---------------+-------------+------------+-------------------+



In [13]:
print("isActive:", debug_query.isActive)
print("status:", debug_query.status)
print("lastProgress:", debug_query.lastProgress)
print("recentProgress:", debug_query.recentProgress[-3:])
print("exception:", debug_query.exception())


ConnectionRefusedError: [Errno 111] Connection refused